# 07 Change Detection — Core Skill

**Курс:** Computer Vision Engineering for DefenceTech  
**Автор:** Serhii Polishchuk  

Кроки: пари кадрів → align → diff mask → events → метрики → FP-таксономія → Note 07.

Конфіг: `configs/change_run.yaml`

## 0. Setup

In [ ]:
!pip install -q opencv-python-headless pyyaml matplotlib pandas

import os, json, random, shutil, zipfile, urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path(".")
CFG_PATH = ROOT / "configs" / "change_run.yaml"
# у Colab скопіюй change_run.yaml у /content/configs/ або вкажи шлях
if not CFG_PATH.exists():
    CFG_PATH = Path("/content/configs/change_run.yaml")
if not CFG_PATH.exists():
    # fallback defaults
    CFG = {
        "seed": 42,
        "align": {"max_features": 2000, "good_match_ratio": 0.75,
                   "ransac_reproj_threshold": 3.0, "min_inliers": 25},
        "diff": {"tau": 25, "blur_ksize": 5, "opening_ksize": 3, "thr_area": 80},
        "events": {"min_area": 80, "max_events_per_pair": 50},
        "track_link": {"iou_threshold": 0.30},
        "eval": {"tau_grid": [10, 15, 20, 25, 30, 40, 50], "proxy_from_tracks": True},
    }
    Path("configs").mkdir(exist_ok=True)
    with open("configs/change_run.yaml", "w") as f:
        yaml.dump(CFG, f, default_flow_style=False)
    CFG_PATH = Path("configs/change_run.yaml")

with open(CFG_PATH) as f:
    CFG = yaml.safe_load(f)

SEED = CFG.get("seed", 42)
random.seed(SEED)
np.random.seed(SEED)

OUT = Path("./change_out")
OUT.mkdir(exist_ok=True)
(OUT / "diff").mkdir(exist_ok=True)
(OUT / "mask").mkdir(exist_ok=True)
(OUT / "events").mkdir(exist_ok=True)

print("Config:", CFG_PATH.resolve())
print(yaml.dump(CFG, default_flow_style=False))

## 1. Дані: послідовність і пари кадрів

- Частина пар з **помітним рухом** об’єктів  
- Частина зі **статичною** сценою (короткий інтервал)  
- Синтетичні пари освітлення (gain/brightness) — контрольовані FP

In [ ]:
MOT_ROOT = Path("./datasets/VisDrone-MOT")
DET_VAL = Path("./datasets/VisDrone/VisDrone2019-DET-val/images")

def find_sequences(root: Path):
    seq_dir = root / "sequences"
    if not seq_dir.exists():
        return []
    return [p for p in sorted(seq_dir.iterdir()) if p.is_dir() and any(p.glob("*.jpg"))]

seqs = find_sequences(MOT_ROOT)

# Якщо MOT немає — demo з DET val (для відладки; для здачі краще MOT)
if not seqs:
    if not DET_VAL.exists():
        # download DET val
        DATA_ROOT = Path("./datasets/VisDrone")
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
        zpath = DATA_ROOT / "VisDrone2019-DET-val.zip"
        if not DET_VAL.exists():
            print("Downloading DET val...")
            urllib.request.urlretrieve(url, zpath)
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(DATA_ROOT)
            zpath.unlink(missing_ok=True)
    demo = MOT_ROOT / "sequences" / "demo_det_val"
    demo.mkdir(parents=True, exist_ok=True)
    imgs = sorted(DET_VAL.glob("*.jpg"))[:150]
    for i, ip in enumerate(imgs, 1):
        dst = demo / f"{i:07d}.jpg"
        if not dst.exists():
            shutil.copy(ip, dst)
    seqs = [demo]
    print("Demo sequence from DET val:", demo)

SEQ = seqs[0]
FRAMES = sorted(SEQ.glob("*.jpg"))
print(f"Sequence: {SEQ.name}, frames: {len(FRAMES)}")

In [ ]:
def load_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def apply_brightness(img_bgr, delta):
    return np.clip(img_bgr.astype(np.int16) + int(delta), 0, 255).astype(np.uint8)

def apply_gain(img_bgr, gain):
    return np.clip(img_bgr.astype(np.float32) * float(gain), 0, 255).astype(np.uint8)

# --- Пари ---
# motion: кадри з кроком (більша зміна сцени)
# static: сусідні кадри
# synth: той самий кадр + brightness/gain (очікуємо FP від освітлення, без реальної зміни об'єктів)

pairs = []  # list of dicts

n = len(FRAMES)
# static pairs (короткий інтервал)
for i in [0, 10, 20]:
    if i + 1 < n:
        pairs.append({
            "id": f"static_{i}_{i+1}",
            "kind": "static",
            "t0": FRAMES[i], "t1": FRAMES[i + 1],
            "expect_change": False,  # proxy: мало змін
        })

# motion pairs (більший крок)
for i in [0, 5, 15]:
    j = min(i + 8, n - 1)
    if j > i:
        pairs.append({
            "id": f"motion_{i}_{j}",
            "kind": "motion",
            "t0": FRAMES[i], "t1": FRAMES[j],
            "expect_change": True,
        })

# synthetic lighting on same frame (controlled FP)
base_idx = min(30, n - 1)
base = FRAMES[base_idx]
img_b, _ = load_gray(base)
for name, fn in [
    ("bright_p40", lambda im: apply_brightness(im, 40)),
    ("bright_m40", lambda im: apply_brightness(im, -40)),
    ("gain_1p2", lambda im: apply_gain(im, 1.2)),
]:
    synth_path = OUT / f"synth_{name}.jpg"
    cv2.imwrite(str(synth_path), fn(img_b))
    pairs.append({
        "id": f"synth_{name}",
        "kind": "synth_light",
        "t0": base, "t1": synth_path,
        "expect_change": False,  # справжніх об'єктних змін немає
        "fp_source": "illumination",
    })

print(f"Pairs: {len(pairs)}")
for p in pairs:
    print(f"  {p['id']:20s} kind={p['kind']:12s} expect_change={p['expect_change']}")

## 2. Align / Diff / Events

- `align_t0_to_t1` — ORB + BFMatcher + RANSAC homography, warp t0 → t1  
- `frame_diff_mask` — |t1 − warped_t0|, threshold τ, opening, area filter  
- `build_change_events` — connected components → JSON-події

In [ ]:
def align_t0_to_t1(img0_bgr, img1_bgr, cfg=None):
    """Register img0 to img1. Returns warped0, H, n_inliers, mean_residual."""
    cfg = cfg or CFG["align"]
    g0 = cv2.cvtColor(img0_bgr, cv2.COLOR_BGR2GRAY)
    g1 = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)

    orb = cv2.ORB_create(cfg["max_features"])
    k0, d0 = orb.detectAndCompute(g0, None)
    k1, d1 = orb.detectAndCompute(g1, None)
    if d0 is None or d1 is None or len(k0) < 8 or len(k1) < 8:
        return img0_bgr.copy(), None, 0, float("inf")

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    knn = bf.knnMatch(d0, d1, k=2)
    good = []
    for m_n in knn:
        if len(m_n) < 2:
            continue
        m, n = m_n
        if m.distance < cfg["good_match_ratio"] * n.distance:
            good.append(m)

    if len(good) < 8:
        return img0_bgr.copy(), None, 0, float("inf")

    src = np.float32([k0[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, cfg["ransac_reproj_threshold"])
    if H is None or mask is None:
        return img0_bgr.copy(), None, 0, float("inf")

    inliers = int(mask.sum())
    # residual on inliers
    if inliers > 0:
        src_i = src[mask.ravel() == 1]
        dst_i = dst[mask.ravel() == 1]
        proj = cv2.perspectiveTransform(src_i, H)
        resid = np.linalg.norm(proj.reshape(-1, 2) - dst_i.reshape(-1, 2), axis=1)
        mean_res = float(resid.mean())
    else:
        mean_res = float("inf")

    h, w = g1.shape[:2]
    warped = cv2.warpPerspective(img0_bgr, H, (w, h))
    return warped, H, inliers, mean_res


def frame_diff_mask(warped0_bgr, img1_bgr, tau=None, cfg=None):
    """Absolute difference → binary change mask."""
    cfg = cfg or CFG["diff"]
    tau = cfg["tau"] if tau is None else tau
    k = cfg["blur_ksize"]
    if k % 2 == 0:
        k += 1
    g0 = cv2.cvtColor(warped0_bgr, cv2.COLOR_BGR2GRAY)
    g1 = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)
    if k > 1:
        g0 = cv2.GaussianBlur(g0, (k, k), 0)
        g1 = cv2.GaussianBlur(g1, (k, k), 0)
    diff = cv2.absdiff(g0, g1)
    _, bw = cv2.threshold(diff, int(tau), 255, cv2.THRESH_BINARY)
    ok = cfg["opening_ksize"]
    if ok > 0:
        if ok % 2 == 0:
            ok += 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ok, ok))
        bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    return diff, bw


def build_change_events(mask, pair_id, thr_area=None, cfg=None):
    """Connected components → list of event dicts."""
    cfg = cfg or CFG["events"]
    thr_area = CFG["diff"]["thr_area"] if thr_area is None else thr_area
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    events = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area < thr_area:
            continue
        cx, cy = centroids[i]
        events.append({
            "pair_id": pair_id,
            "component_id": int(i),
            "bbox": [int(x), int(y), int(x + w), int(y + h)],
            "area": int(area),
            "centroid": [float(cx), float(cy)],
            "track_id": None,
        })
        if len(events) >= cfg["max_events_per_pair"]:
            break
    return events

print("Functions ready: align_t0_to_t1, frame_diff_mask, build_change_events")

In [ ]:
# --- Прогін усіх пар ---
MIN_INLIERS = CFG["align"]["min_inliers"]
results = []
rejected = []

for p in tqdm(pairs, desc="pairs"):
    img0, _ = load_gray(p["t0"])
    img1, _ = load_gray(p["t1"])
    # resize to same size if needed
    if img0.shape[:2] != img1.shape[:2]:
        img0 = cv2.resize(img0, (img1.shape[1], img1.shape[0]))

    warped, H, inliers, mean_res = align_t0_to_t1(img0, img1)
    rec = {
        "pair_id": p["id"],
        "kind": p["kind"],
        "expect_change": p["expect_change"],
        "inliers": inliers,
        "mean_residual": mean_res,
        "registered": H is not None and inliers >= MIN_INLIERS,
        "fp_source": p.get("fp_source"),
    }

    if not rec["registered"]:
        rec["reject_reason"] = f"inliers={inliers} < min={MIN_INLIERS}"
        rejected.append(rec)
        results.append(rec)
        print(f"REJECT {p['id']}: {rec['reject_reason']}")
        continue

    diff, mask = frame_diff_mask(warped, img1)
    events = build_change_events(mask, p["id"])
    rec["n_events"] = len(events)
    rec["events"] = events

    # save visuals
    cv2.imwrite(str(OUT / "diff" / f"{p['id']}_diff.png"), diff)
    cv2.imwrite(str(OUT / "mask" / f"{p['id']}_mask.png"), mask)
    with open(OUT / "events" / f"{p['id']}.json", "w") as f:
        json.dump({"pair_id": p["id"], "events": events}, f, indent=2)

    results.append(rec)
    print(f"OK {p['id']}: inliers={inliers}, residual={mean_res:.2f}, events={len(events)}")

df_pairs = pd.DataFrame([{k: r.get(k) for k in
    ["pair_id", "kind", "expect_change", "inliers", "mean_residual", "registered", "n_events", "reject_reason"]}
    for r in results])
display(df_pairs)
df_pairs.to_csv(OUT / "pairs_summary.csv", index=False)

## 3. Крива Precision / Recall / F1 за τ

Проксі-розмітка: для `expect_change=True` вважаємо, що має бути ≥1 подія (TP, якщо є);  
для `expect_change=False` будь-яка подія = FP.  
(Груба проксі без pixel-GT змін; у Note 07 це обмеження.)

In [ ]:
tau_grid = CFG["eval"]["tau_grid"]
curve = []

for tau in tau_grid:
    TP = FP = FN = 0
    for p in pairs:
        # skip rejected alignment
        meta = next((r for r in results if r["pair_id"] == p["id"]), None)
        if meta is None or not meta.get("registered", False):
            continue
        img0, _ = load_gray(p["t0"])
        img1, _ = load_gray(p["t1"])
        if img0.shape[:2] != img1.shape[:2]:
            img0 = cv2.resize(img0, (img1.shape[1], img1.shape[0]))
        warped, H, inl, _ = align_t0_to_t1(img0, img1)
        if H is None or inl < MIN_INLIERS:
            continue
        _, mask = frame_diff_mask(warped, img1, tau=tau)
        ev = build_change_events(mask, p["id"])
        has = len(ev) > 0
        if p["expect_change"]:
            if has:
                TP += 1
            else:
                FN += 1
        else:
            if has:
                FP += 1
            # TN not counted for P/R here
    prec = TP / max(TP + FP, 1)
    rec = TP / max(TP + FN, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    curve.append({"tau": tau, "TP": TP, "FP": FP, "FN": FN,
                  "precision": round(prec, 3), "recall": round(rec, 3), "F1": round(f1, 3)})

df_curve = pd.DataFrame(curve)
display(df_curve)

best = df_curve.loc[df_curve["F1"].idxmax()]
print(f"\nРобочий поріг (max F1): tau={int(best['tau'])}  F1={best['F1']}  P={best['precision']}  R={best['recall']}")
print("Обґрунтування: баланс FP (хибні тривоги на static/synth) і FN (пропуск motion).")

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(df_curve["tau"], df_curve["precision"], marker="o", label="precision")
ax.plot(df_curve["tau"], df_curve["recall"], marker="s", label="recall")
ax.plot(df_curve["tau"], df_curve["F1"], marker="^", label="F1")
ax.axvline(best["tau"], color="gray", ls="--", label=f"tau*={int(best['tau'])}")
ax.set_xlabel("tau"); ax.set_ylabel("score"); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("P / R / F1 vs tau (proxy labels)")
plt.tight_layout()
plt.savefig(OUT / "pr_f1_curve.png", dpi=120)
plt.show()

df_curve.to_csv(OUT / "pr_f1_curve.csv", index=False)

## 4. Хибні тривоги — таксономія + diff/mask

In [ ]:
# Показати synth (illumination FP) і один motion (справжня зміна)
show_ids = [p["id"] for p in pairs if p["kind"] in ("synth_light", "motion")][:4]

fig, axes = plt.subplots(len(show_ids), 3, figsize=(12, 3 * len(show_ids)))
if len(show_ids) == 1:
    axes = np.array([axes])

for row, pid in enumerate(show_ids):
    p = next(x for x in pairs if x["id"] == pid)
    img0, _ = load_gray(p["t0"])
    img1, _ = load_gray(p["t1"])
    if img0.shape[:2] != img1.shape[:2]:
        img0 = cv2.resize(img0, (img1.shape[1], img1.shape[0]))
    warped, H, inl, _ = align_t0_to_t1(img0, img1)
    if H is None:
        continue
    diff, mask = frame_diff_mask(warped, img1, tau=int(best["tau"]))
    axes[row, 0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"{pid} t1")
    axes[row, 1].imshow(diff, cmap="gray")
    axes[row, 1].set_title("diff")
    axes[row, 2].imshow(mask, cmap="gray")
    axes[row, 2].set_title("mask")
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle("FP taxonomy: synth_light vs motion (diff/mask)")
plt.tight_layout()
plt.savefig(OUT / "fp_examples.png", dpi=120)
plt.show()

print("""
Таксономія хибних тривог (джерела):
1. illumination — синтетичний gain/brightness (контрольовані FP)
2. residual_registration — залишки після warp (паралакс / мало inliers)
3. texture_noise — дрібна текстура, вітер листя (якщо є на MOT)
4. true_change — motion-пари (для порівняння, не FP)
""")

## 5. Формат події (JSON) + опційний track_id

In [ ]:
def iou_box(a, b):
    # a,b = [x1,y1,x2,y2]
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

# Приклад події
example_events = []
for r in results:
    if r.get("events"):
        example_events = r["events"][:2]
        break

example_json = {
    "pair_id": example_events[0]["pair_id"] if example_events else "motion_0_8",
    "tau": int(best["tau"]),
    "events": example_events or [{
        "component_id": 1,
        "bbox": [100, 120, 130, 165],
        "area": 1350,
        "centroid": [115.0, 142.5],
        "track_id": None,
    }],
}
print(json.dumps(example_json, indent=2, ensure_ascii=False))
with open(OUT / "event_example.json", "w") as f:
    json.dump(example_json, f, indent=2, ensure_ascii=False)

print("\nОпційний track_id: якщо є bbox треків на t1, призначаємо track_id при IoU >=",
      CFG["track_link"]["iou_threshold"])
print("(Підключити events з DZ05 pipeline_out/*_events.csv за потреби.)")

## 6. Чернетка Note 07

Після прогону заповни/експортуй:

| Блок | Звідки |
|------|--------|
| Вхідні пари | `pairs_summary.csv` |
| Метод | align ORB+H, absdiff, opening, CC |
| YAML | `configs/change_run.yaml` |
| Метрики | `pr_f1_curve.csv` + графік |
| FP | synth illumination + residual |
| Подія | `event_example.json` |
| L6 | track_id опційно через IoU |
| Обмеження | проксі-розмітка, паралакс, освітлення, demo≠MOT |